# Trimmed Source ↔ Entity Graph — Gephi Export

Same idea as `metatopic_trimmed_gephi_export.ipynb`, but centre nodes are
**source sub-units** (news outlet / talkshow programme / kamer doc type)
instead of meta-topics. All three arenas are included — sub-units don't
require `topic_meta`, so kamer rejoins here.

Two curated entity sets per sub-unit centre:
- **distinctive** entities (the fans): top-N per sub-unit by TF-IDF
- **shared** entities (the centre): top-M globally by how many sub-units they span

**Cleaning** reuses the stage-one pipeline: normalisation, noise filter
(empty/single-char/numeric), alias merge via the review workbook's `canonical`
column for `keep="yes"` rows, and drop everything `keep="no"` plus the small
junk stop-list.

In [23]:
import ast
import re
import pathlib
import numpy as np
import pandas as pd

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS
# ============================================================

SOURCES = {
    "news": {
        "path":        "../../news/analysis/df_with_NER.csv",
        "subunit_col": "outlet",
        "node_type":   "news",
    },
    "talkshows": {
        "path":        "../../subtitles/analysis/subs_with_NER.csv",
        "subunit_col": "program",
        "node_type":   "talkshows",
    },
    "kamer": {
        "path":        "../../tweede_kamer/analysis/Tweede_Kamer_with_NER.csv",
        "subunit_col": "type",
        "node_type":   "kamer",
    },
}

ENTITY_COLS = {
    "persons":   "PER",
    "orgs":      "ORG",
    "countries": "LOC",
}

REVIEW_XLSX   = "entity_review.xlsx"        # three-sheet workbook (PER/ORG/LOC); optional
JUNK_STOPLIST = {"wie", "dat.", "anders"}    # normalised lowercase; always dropped

TOP_N_DISTINCTIVE = 50     # top entities per sub-unit by TF-IDF
TOP_M_SHARED       = 70    # top entities globally by sub-unit spread
MIN_SUBUNIT_FREQ     = 3   # drop (entity, sub-unit) pairs below this before ranking
MIN_EDGE_WEIGHT      = 0.01  # coverage-share floor for drawing an edge

OUT_NODES         = "source_trim_nodes_50.csv"
OUT_EDGES         = "source_trim_edges_50.csv"
OUT_SUBUNITS_XLSX = "source_trim_subunits.xlsx"
# ============================================================

## 1. Helpers

In [24]:
def parse_entity_list(cell):
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if s in ("", "[]", "nan"):
        return []
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [str(x) for x in result]
        return [str(result)]
    except (ValueError, SyntaxError):
        s = re.sub(r"^[\[\(]|[\]\)]$", "", s)
        return [x.strip().strip("'\"" ) for x in s.split(",") if x.strip()]


def normalise(s):
    return re.sub(r"\s+", " ", str(s).strip())


def slugify(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


def make_unique_id(prefix, label, used_ids):
    base = slugify(label) or "unk"
    candidate = f"{prefix}_{base}"
    if candidate not in used_ids:
        used_ids.add(candidate)
        return candidate
    i = 2
    while f"{candidate}_{i}" in used_ids:
        i += 1
    final = f"{candidate}_{i}"
    used_ids.add(final)
    return final

## 2. Alias/drop lookup from review workbook (if present)

In [25]:
entity_to_canonical = {}
canonical_to_type_review = {}
drop_set = set(JUNK_STOPLIST)

review_path = (NB_DIR / REVIEW_XLSX).resolve()
if review_path.exists():
    for sheet in ["PER", "ORG", "LOC"]:
        sheet_df = pd.read_excel(review_path, sheet_name=sheet)
        for _, row in sheet_df.iterrows():
            key = normalise(row["entity"]).lower()
            if str(row["keep"]).strip().lower() == "no":
                drop_set.add(key)
    for sheet in ["PER", "ORG", "LOC"]:
        sheet_df = pd.read_excel(review_path, sheet_name=sheet)
        kept = sheet_df[sheet_df["keep"].astype(str).str.strip().str.lower() == "yes"]
        for _, row in kept.iterrows():
            key = normalise(row["entity"]).lower()
            if key in drop_set:
                continue
            canonical = normalise(row["canonical"])
            entity_to_canonical[key] = canonical
            canonical_to_type_review[canonical] = sheet
    print(f"Review workbook found: {len(entity_to_canonical)} alias mappings, "
          f"{len(drop_set)} dropped keys (incl. junk stop-list)")
else:
    print("No review workbook found — only junk stop-list applied.")

Review workbook found: 764 alias mappings, 136 dropped keys (incl. junk stop-list)


## 3. Load all three arenas

In [26]:
frames = {}

for arena, cfg in SOURCES.items():
    abs_path = (NB_DIR / cfg["path"]).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {arena}: not found")
        continue
    df = pd.read_csv(abs_path)
    df[cfg["subunit_col"]] = df[cfg["subunit_col"]].fillna("UNKNOWN").astype(str).str.strip()
    frames[arena] = df
    print(f"[OK] {arena}: {len(df):,} docs, "
          f"{df[cfg['subunit_col']].nunique()} sub-units ({cfg['subunit_col']})")

[OK] news: 13,209 docs, 8 sub-units (outlet)
[OK] talkshows: 495 docs, 9 sub-units (program)
[OK] kamer: 844 docs, 3 sub-units (type)


## 4. Parse all documents → long format

In [27]:
records = []
for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    df = frames[arena]
    subunit_col = cfg["subunit_col"]
    for row_idx, row in df.iterrows():
        subunit = row[subunit_col]
        doc_id  = f"{arena}:{row_idx}"
        doc_seen = {}   # canonical -> raw_type (first seen, deduped within doc)
        for col, raw_type in ENTITY_COLS.items():
            for raw in parse_entity_list(row.get(col)):
                norm = normalise(raw)
                if len(norm) <= 1 or norm.isdigit():
                    continue
                key = norm.lower()
                if key in drop_set:
                    continue
                canonical = entity_to_canonical.get(key, norm)
                if canonical not in doc_seen:
                    doc_seen[canonical] = raw_type
        for canonical, raw_type in doc_seen.items():
            records.append({
                "subunit":   subunit,
                "doc_id":    doc_id,
                "canonical": canonical,
                "raw_type":  raw_type,
            })

long_df = pd.DataFrame(records)
print(f"(doc, canonical) pairings: {len(long_df):,}")
print(f"Distinct canonical entities: {long_df['canonical'].nunique():,}")

(doc, canonical) pairings: 211,111
Distinct canonical entities: 82,291


## 5. Resolve entity_type, sub-unit doc totals, and the freq table

In [28]:
# entity_type: review-corrected wins, else doc-level majority vote
type_doc = long_df[["canonical", "doc_id", "raw_type"]].drop_duplicates()
type_counts = type_doc.groupby(["canonical", "raw_type"]).size().reset_index(name="n")
computed_dominant_type = (
    type_counts.loc[type_counts.groupby("canonical")["n"].idxmax()]
    .set_index("canonical")["raw_type"]
)
canonical_type = {
    c: canonical_to_type_review.get(c, computed_dominant_type.get(c, ""))
    for c in long_df["canonical"].unique()
}

# total documents per sub-unit (coverage-share denominator) + arena node_type
subunit_doc_totals = {}
subunit_node_type  = {}
for arena, cfg in SOURCES.items():
    if arena not in frames:
        continue
    for subunit, grp in frames[arena].groupby(cfg["subunit_col"]):
        subunit_doc_totals[subunit] = len(grp)
        subunit_node_type[subunit]  = cfg["node_type"]
total_subunits = len(subunit_doc_totals)
print(f"Total sub-units: {total_subunits}")

# freq_full: distinct docs per (subunit, entity) — used later for edge weights (no floor)
freq_full = long_df.groupby(["subunit", "canonical"])["doc_id"].nunique()

# freq_filtered: candidate pool for TF-IDF / shared ranking
freq_filtered = freq_full[freq_full >= MIN_SUBUNIT_FREQ]
print(f"(subunit, entity) pairs: {len(freq_full):,} total, "
      f"{len(freq_filtered):,} after min_subunit_freq={MIN_SUBUNIT_FREQ}")

Total sub-units: 20
(subunit, entity) pairs: 111,473 total, 10,476 after min_subunit_freq=3


## 6. TF-IDF and entity selection

- **distinctive**: top `top_n_distinctive` per sub-unit by `tf * idf`
- **shared**: top `top_m_shared` globally by sub-unit spread (tie-break: total freq)
- on overlap, **shared wins** (no duplicate node)

In [29]:
subunits_per_entity = freq_filtered.groupby("canonical").size()
idf = np.log(total_subunits / subunits_per_entity)

tfidf_df = freq_filtered.reset_index()
tfidf_df.columns = ["subunit", "canonical", "freq"]
tfidf_df["idf"]   = tfidf_df["canonical"].map(idf)
tfidf_df["tfidf"] = tfidf_df["freq"] * tfidf_df["idf"]

# --- distinctive: top N per sub-unit ---
distinctive_pairs = (
    tfidf_df.sort_values(["subunit", "tfidf"], ascending=[True, False])
    .groupby("subunit")
    .head(TOP_N_DISTINCTIVE)
    .copy()
)
distinctive_entities = set(distinctive_pairs["canonical"])

# --- shared: top M globally by sub-unit spread, tie-break total freq ---
entity_stats = (
    tfidf_df.groupby("canonical")
    .agg(n_subunits=("subunit", "nunique"), total_freq=("freq", "sum"))
    .sort_values(["n_subunits", "total_freq"], ascending=[False, False])
)
shared_entities = set(entity_stats.head(TOP_M_SHARED).index)

# --- combine, shared takes precedence ---
final_roles = {e: "distinctive" for e in distinctive_entities}
final_roles.update({e: "shared" for e in shared_entities})

n_shared = sum(1 for v in final_roles.values() if v == "shared")
n_distinctive = sum(1 for v in final_roles.values() if v == "distinctive")
print(f"Distinctive entities (pure): {n_distinctive}")
print(f"Shared entities: {n_shared}")
print(f"Total selected entities: {len(final_roles)}")

Distinctive entities (pure): 372
Shared entities: 70
Total selected entities: 442


## 7. Build nodes

In [30]:
entity_total_docs = long_df.groupby("canonical")["doc_id"].nunique()

# best-scoring sub-unit per distinctive entity (for dominant_source on overlap)
best_subunit_for_entity = (
    distinctive_pairs.loc[distinctive_pairs.groupby("canonical")["tfidf"].idxmax()]
    .set_index("canonical")["subunit"]
)

used_ids = set()
subunit_node_rows = [
    {
        "Id":             make_unique_id("src", s, used_ids),
        "Label":          s,
        "node_type":      subunit_node_type[s],
        "entity_type":    "",
        "role":           "",
        "size":           total,
        "dominant_source": s,
    }
    for s, total in subunit_doc_totals.items()
]
subunit_id = {r["Label"]: r["Id"] for r in subunit_node_rows}

entity_node_rows = [
    {
        "Id":             make_unique_id("ent", e, used_ids),
        "Label":          e,
        "node_type":      "entity",
        "entity_type":    canonical_type.get(e, ""),
        "role":           role,
        "size":           int(entity_total_docs.get(e, 0)),
        "dominant_source": "shared" if role == "shared" else best_subunit_for_entity.get(e, ""),
    }
    for e, role in final_roles.items()
]
entity_id = {r["Label"]: r["Id"] for r in entity_node_rows}

nodes_df = pd.DataFrame(subunit_node_rows + entity_node_rows)
dupes = nodes_df[nodes_df.duplicated("Id", keep=False)]
if len(dupes):
    print(f"WARNING: duplicate node Ids: {dupes['Id'].tolist()[:10]}")
else:
    print("Node Ids: no collisions")
print(f"Sub-unit nodes: {len(subunit_node_rows)}, entity nodes: {len(entity_node_rows)}")

Node Ids: no collisions
Sub-unit nodes: 20, entity nodes: 442


## 8. Build edges

For each selected entity, edges go to **every** sub-unit where its coverage
share (using `freq_full`, not the `min_subunit_freq`-filtered table) exceeds
`min_edge_weight`. Distinctive entities will mostly produce one edge; shared
entities will produce many.

In [31]:
edge_rows = []
for e in final_roles:
    for subunit, total in subunit_doc_totals.items():
        f = freq_full.get((subunit, e), 0)
        if f == 0:
            continue
        share = f / total
        if share <= MIN_EDGE_WEIGHT:
            continue
        edge_rows.append({
            "Source": subunit_id[subunit],
            "Target": entity_id[e],
            "Weight": round(share, 6),
            "Type":   "Undirected",
        })

edges_df = pd.DataFrame(edge_rows)
print(f"Edges: {len(edges_df)}")

Edges: 1144


## 9. Export

In [32]:
nodes_path = NB_DIR / OUT_NODES
edges_path = NB_DIR / OUT_EDGES
nodes_df.to_csv(nodes_path, index=False)
edges_df.to_csv(edges_path, index=False)

# per-sub-unit readable table: the raw top-N TF-IDF ranking per sub-unit
per_subunit = distinctive_pairs.copy()
per_subunit["entity_type"] = per_subunit["canonical"].map(canonical_type)
per_subunit["final_role"]  = per_subunit["canonical"].map(final_roles)
per_subunit = per_subunit.rename(columns={
    "subunit": "source_subunit", "canonical": "entity",
    "freq": "freq_in_subunit", "tfidf": "tfidf_score",
})
per_subunit = per_subunit[["source_subunit", "entity", "entity_type", "freq_in_subunit", "tfidf_score", "final_role"]]
per_subunit["tfidf_score"] = per_subunit["tfidf_score"].round(3)

subunits_path = NB_DIR / OUT_SUBUNITS_XLSX
per_subunit.to_excel(subunits_path, index=False)

print(f"Saved -> {nodes_path.name}, {edges_path.name}, {subunits_path.name}")

Saved -> source_trim_nodes_50.csv, source_trim_edges_50.csv, source_trim_subunits.xlsx


## 10. Summary

In [33]:
entity_degree = edges_df.groupby("Target")["Source"].nunique()

print("=" * 55)
print("SUMMARY")
print(f"  Distinctive entities (pure): {n_distinctive}")
print(f"  Shared entities:             {n_shared}")
print(f"  Sub-unit nodes:               {len(subunit_node_rows)}")
print(f"  Total edges:                 {len(edges_df)}")
print()

id_to_label = dict(zip(nodes_df["Id"], nodes_df["Label"]))
shared_ids = {entity_id[e] for e in shared_entities}
distinctive_only_ids = {entity_id[e] for e in final_roles if final_roles[e] == "distinctive"}

print("Top 5 shared entities (highest degree):")
for eid, deg in entity_degree[entity_degree.index.isin(shared_ids)].sort_values(ascending=False).head(5).items():
    print(f"  {id_to_label[eid]:<25s} degree={deg}")
print()
print("Sample distinctive entities (degree=1, the 'fans'):")
for eid, deg in entity_degree[entity_degree.index.isin(distinctive_only_ids)].sort_values().head(5).items():
    print(f"  {id_to_label[eid]:<25s} degree={deg}")
print("=" * 55)

SUMMARY
  Distinctive entities (pure): 372
  Shared entities:             70
  Sub-unit nodes:               20
  Total edges:                 1144

Top 5 shared entities (highest degree):
  Nederland                 degree=19
  Google                    degree=19
  VS                        degree=18
  Duitsland                 degree=17
  België                    degree=16

Sample distinctive entities (degree=1, the 'fans'):
  VPRO                      degree=1
  50PLUS                    degree=1
  Victoria Warmerdam        degree=1
  Victoria                  degree=1
  Verhoeven                 degree=1
